# 暂退法（Dropout）

## 1. 为什么需要 Dropout？

深度神经网络具有较强的拟合能力，但也容易出现**过拟合**：训练集表现很好，测试集表现较差。

造成过拟合的原因包括：

- 模型过于复杂，参数过多。
- 训练数据不足。
- 神经元过度依赖某些特征。

**Dropout 是一种正则化技术，通过在训练过程中随机丢弃部分神经元，降低模型对特定特征的依赖，从而缓解过拟合。**

## 2. Dropout 的基本原理

在神经网络训练过程中，Dropout 会以一定概率 $p$ 将部分神经元的输出置为 0。

例如，隐藏层有 5 个神经元，当 `dropout=0.4` 时，每个神经元都有 40% 的概率被丢弃。

- **训练阶段**：随机丢弃部分神经元。
- **测试阶段**：保留全部神经元，不执行 Dropout。

这样可以减少神经元之间的过度依赖，提高模型的泛化能力。

## 3. Dropout 的数学公式

设 $h$ 为神经元原始输出，$p$ 为丢弃概率：

$$
h'=
\begin{cases}
0, & \text{概率为 }p \\[6pt]
\frac{h}{1-p}, & \text{概率为 }1-p
\end{cases}
$$

其中：

- $p$：神经元被丢弃的概率。
- $1-p$：神经元被保留的概率。
- $\frac{h}{1-p}$：对保留的神经元进行缩放。

**为什么需要除以 $1-p$？**

为了保证 Dropout 前后输出的期望值保持一致：

$$
E[h']=p\times0+(1-p)\times\frac{h}{1-p}=h
$$

## 4. Dropout 的实现思路

1. 生成与输入张量形状相同的随机数，服从均匀分布 $U(0,1)$。
2. 将随机数小于 $p$ 的位置置为 0。
3. 保留其余位置的元素，并除以 $1-p$。

在 PyTorch 中可以直接使用：

```python
nn.Dropout(p=0.5)
```

表示训练时每个元素有 50% 的概率被置为 0。

## 5. 总结

**Dropout 的核心思想：训练时随机丢弃部分神经元，减少模型对少数特征的依赖，从而缓解过拟合。**

需要注意：

- Dropout 只在训练模式下生效。
- `p` 越大，丢弃比例越高，但过大可能导致欠拟合。
- Dropout 不能保证一定提升模型性能，需要结合实际数据调整。

In [ ]:
import torch  # PyTorch深度学习框架
from torch import nn  # 神经网络模块
from d2l import torch as d2l  # 动手学深度学习工具包


def dropout_layer(X, dropout):
    """实现暂退法（Dropout）"""
    assert 0 <= dropout <= 1  # 确保丢弃概率在[0, 1]之间

    if dropout == 1:
        return torch.zeros_like(X)  # 丢弃所有元素，全部置为0

    if dropout == 0:
        return X  # 保留所有元素，不做任何处理

    # 生成随机掩码：大于dropout的位置为1，否则为0
    # mask = (torch.rand(X.shape) > dropout).float()
    mask = (torch.rand_like(X) > dropout).float()

    # 丢弃部分元素，并缩放保留的元素以保持期望值不变
    return mask * X / (1.0 - dropout)

我们可以通过下面几个例子来[**测试`dropout_layer`函数**]。
我们将输入`X`通过暂退法操作，暂退概率分别为0、0.5和1。


In [ ]:
# 创建一个2行8列的张量，元素为0~15
X = torch.arange(16, dtype=torch.float32).reshape((2, 8))

print(X)  # 输出原始张量

print(dropout_layer(X, 0.))   # 丢弃概率为0，保留所有元素

print(dropout_layer(X, 0.5))  # 丢弃概率为0.5，随机丢弃约50%的元素，并将保留值乘以2

print(dropout_layer(X, 1.))   # 丢弃概率为1，所有元素置为0

### 定义模型参数

同样，我们使用 :numref:`sec_fashion_mnist`中引入的Fashion-MNIST数据集。
我们[**定义具有两个隐藏层的多层感知机，每个隐藏层包含256个单元**]。


In [ ]:
# 定义输入层、输出层及两个隐藏层的神经元数量
num_inputs, num_outputs, num_hiddens1, num_hiddens2 = 784, 10, 256, 256

### 定义模型

我们可以将暂退法应用于每个隐藏层的输出（在激活函数之后），
并且可以为每一层分别设置暂退概率：
常见的技巧是在靠近输入层的地方设置较低的暂退概率。
下面的模型将第一个和第二个隐藏层的暂退概率分别设置为0.2和0.5，
并且暂退法只在训练期间有效。


In [ ]:
# 设置两个隐藏层的Dropout丢弃概率
dropout1, dropout2 = 0.2, 0.5


class Net(nn.Module):
    """定义包含两个隐藏层和Dropout的多层感知机"""

    def __init__(self, num_inputs, num_outputs, num_hiddens1, num_hiddens2,
                 is_training=True):
        super(Net, self).__init__()  # 初始化父类

        self.num_inputs = num_inputs  # 输入特征数量
        self.training = is_training   # 是否处于训练模式

        # 定义三个全连接层
        self.lin1 = nn.Linear(num_inputs, num_hiddens1)    # 784 → 256
        self.lin2 = nn.Linear(num_hiddens1, num_hiddens2)  # 256 → 256
        self.lin3 = nn.Linear(num_hiddens2, num_outputs)   # 256 → 10

        self.relu = nn.ReLU()  # 定义ReLU激活函数

    def forward(self, X):
        """定义前向传播过程"""

        # 第一隐藏层：展平输入 → 全连接 → ReLU
        H1 = self.relu(self.lin1(X.reshape((-1, self.num_inputs))))

        if self.training:
            H1 = dropout_layer(H1, dropout1)  # 随机丢弃20%的神经元输出

        # 第二隐藏层：全连接 → ReLU
        H2 = self.relu(self.lin2(H1))

        if self.training:
            H2 = dropout_layer(H2, dropout2)  # 随机丢弃50%的神经元输出

        out = self.lin3(H2)  # 输出10个类别的预测分数
        return out


# 实例化多层感知机模型
net = Net(num_inputs, num_outputs, num_hiddens1, num_hiddens2)

### [**训练和测试**]

这类似于前面描述的多层感知机训练和测试。


In [ ]:
# 设置训练轮数、学习率和批量大小
num_epochs, lr, batch_size = 10, 0.5, 256

# 定义交叉熵损失函数，保留每个样本的损失值
loss = nn.CrossEntropyLoss(reduction='none')

# 加载Fashion-MNIST训练集和测试集
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)

# 使用SGD优化器更新模型参数，学习率为0.5
trainer = torch.optim.SGD(net.parameters(), lr=lr)

# 训练模型，并评估训练集和测试集的准确率
# d2l.train_ch3(net, train_iter, test_iter, loss, num_epochs, trainer)
train_ch3(net, train_iter, test_iter, loss, num_epochs, trainer)

## [**简洁实现**]

对于深度学习框架的高级API，我们只需在每个全连接层之后添加一个`Dropout`层，
将暂退概率作为唯一的参数传递给它的构造函数。
在训练时，`Dropout`层将根据指定的暂退概率随机丢弃上一层的输出（相当于下一层的输入）。
在测试时，`Dropout`层仅传递数据。


In [ ]:
# 使用PyTorch内置模块构建带Dropout的多层感知机
net = nn.Sequential(
    nn.Flatten(),          # 将28×28图像展平为784维向量
    nn.Linear(784, 256),   # 第一隐藏层：784 → 256
    nn.ReLU(),             # ReLU激活函数
    nn.Dropout(dropout1),  # 随机丢弃20%的神经元输出

    nn.Linear(256, 256),   # 第二隐藏层：256 → 256
    nn.ReLU(),             # ReLU激活函数
    nn.Dropout(dropout2),  # 随机丢弃50%的神经元输出

    nn.Linear(256, 10)     # 输出层：256 → 10个类别
)

def init_weights(m):
    """初始化全连接层的权重"""
    if type(m) == nn.Linear:  # 判断是否为全连接层
        nn.init.normal_(m.weight, std=0.01)  # 正态分布初始化权重

net.apply(init_weights)  # 对所有全连接层应用权重初始化

接下来，我们[**对模型进行训练和测试**]。


In [ ]:
# 使用随机梯度下降（SGD）优化器，学习率为lr
trainer = torch.optim.SGD(net.parameters(), lr=lr)

# 训练模型，计算训练损失、训练准确率和测试准确率
train_ch3(net, train_iter, test_iter, loss, num_epochs, trainer)

## 小结

* 暂退法在前向传播过程中，计算每一内部层的同时丢弃一些神经元。
* 暂退法可以避免过拟合，它通常与控制权重向量的维数和大小结合使用的。
* 暂退法将活性值$h$替换为具有期望值$h$的随机变量。
* 暂退法仅在训练期间使用。

[Discussions](https://discuss.d2l.ai/t/1813)
